# Day 24 — Delta Lake Fundamentals

**What you will learn:**
- What Delta Lake adds on top of Parquet (transaction log, ACID)
- Creating and reading Delta tables
- ACID transactions: what they guarantee
- Time travel: reading historical versions and rolling back
- Schema enforcement vs schema evolution
- `DESCRIBE DETAIL`, `DESCRIBE HISTORY`

**Exam relevance:** Delta Lake is core to the Databricks exam — ACID semantics, time travel, and schema enforcement are always tested.

In [ ]:
# Delta Lake is bundled with Databricks Runtime
# In standalone Spark: pip install delta-spark and configure accordingly
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit
import tempfile, os

DELTA_PATH = tempfile.mkdtemp(prefix="delta_lab_")
print(f"Delta table path: {DELTA_PATH}")

## 1. What Is Delta Lake?

Delta Lake = Parquet files + `_delta_log/` directory

```
mytable/
├── _delta_log/
│   ├── 00000000000000000000.json   ← commit 0: CREATE TABLE
│   ├── 00000000000000000001.json   ← commit 1: INSERT
│   ├── 00000000000000000002.json   ← commit 2: UPDATE
│   └── 00000000000000000010.checkpoint.parquet  ← checkpoint
├── part-00000-abc.snappy.parquet
└── part-00001-def.snappy.parquet
```

**The transaction log enables:**
- ACID transactions
- Time travel (read any previous version)
- Audit history
- Concurrent reads and writes without corruption

**ACID guarantees:**
| Property | Meaning |
|---|---|
| **A**tomicity | Write succeeds fully or not at all |
| **C**onsistency | Schema and constraints always satisfied |
| **I**solation | Concurrent transactions don't corrupt each other |
| **D**urability | Committed data survives crashes |

## 2. Creating a Delta Table

In [ ]:
data_v0 = [
    (1, "Alice",  "Engineering", 95000),
    (2, "Bob",    "Marketing",   72000),
    (3, "Carol",  "Engineering", 110000),
    (4, "Dave",   "Sales",       58000),
]
df = spark.createDataFrame(data_v0, ["id", "name", "dept", "salary"])

# Write as Delta — creates _delta_log/ alongside Parquet files
df.write.format("delta").mode("overwrite").save(DELTA_PATH)

print("Delta table created (version 0).")
spark.read.format("delta").load(DELTA_PATH).show()

## 3. Appending and Overwriting

In [ ]:
# Append new rows — creates version 1
new_rows = spark.createDataFrame([
    (5, "Eve",   "Marketing", 81000),
    (6, "Frank", "Engineering", 88000),
], ["id", "name", "dept", "salary"])

new_rows.write.format("delta").mode("append").save(DELTA_PATH)

print("After append (version 1):")
spark.read.format("delta").load(DELTA_PATH).show()

## 4. Time Travel — Read Historical Versions

In [ ]:
# Read version 0 — before the append
print("=== Version 0 (original) ===")
spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(DELTA_PATH) \
    .show()

# Read version 1 — after append
print("=== Version 1 (after append) ===")
spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .load(DELTA_PATH) \
    .show()

# Read by timestamp (any ISO timestamp before a known version)
# spark.read.format("delta").option("timestampAsOf", "2024-01-01").load(DELTA_PATH)

## 5. DeltaTable API — Update, Delete, Merge

In [ ]:
dt = DeltaTable.forPath(spark, DELTA_PATH)

# UPDATE — version 2
dt.update(
    condition=col("dept") == "Sales",
    set={"salary": lit(62000)}
)
print("After UPDATE (version 2):")
spark.read.format("delta").load(DELTA_PATH).show()

In [ ]:
# DELETE — version 3
dt.delete(condition=col("id") == 2)
print("After DELETE (version 3):")
spark.read.format("delta").load(DELTA_PATH).show()

In [ ]:
# MERGE (upsert) — version 4
updates = spark.createDataFrame([
    (1, "Alice",  "Engineering", 100000),  # update salary
    (7, "Grace",  "Sales",       65000),    # new row
], ["id", "name", "dept", "salary"])

dt.alias("target").merge(
    updates.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("After MERGE (version 4):")
spark.read.format("delta").load(DELTA_PATH).orderBy("id").show()

## 6. DESCRIBE HISTORY — Full Audit Log

In [ ]:
# Full transaction history
dt.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

## 7. Schema Enforcement vs Schema Evolution

| Mode | Behaviour |
|---|---|
| **Enforcement** (default) | Writing a DataFrame with extra/incompatible columns → exception |
| **Evolution** | Adding columns allowed with `.option("mergeSchema", "true")` |

In [ ]:
# Schema enforcement — new column raises error
try:
    extra_col = spark.createDataFrame([
        (8, "Hank", "HR", 70000, "London")
    ], ["id", "name", "dept", "salary", "city"])
    extra_col.write.format("delta").mode("append").save(DELTA_PATH)
    print("ERROR: Should have raised an exception")
except Exception as e:
    print(f"Schema enforcement blocked write: {type(e).__name__}")

# Schema evolution — allow new column
extra_col.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(DELTA_PATH)

print("After schema evolution (mergeSchema=true):")
spark.read.format("delta").load(DELTA_PATH).orderBy("id").show()

---

## Day 24 Checklist

- [ ] Know the Delta Lake structure: Parquet files + `_delta_log/`
- [ ] Know ACID: Atomicity, Consistency, Isolation, Durability — what each means
- [ ] Created a Delta table with `format("delta")`
- [ ] Time-travelled with `versionAsOf` and `timestampAsOf`
- [ ] Used `DeltaTable.update()`, `delete()`, `merge()` API
- [ ] Read `DESCRIBE HISTORY` — each operation is a version
- [ ] Know schema enforcement (default) vs evolution (`mergeSchema=true`)

**Next:** Day 25 — Delta Lake Advanced (OPTIMIZE, VACUUM, Z-ORDER, CDF)